# 04 — State B FEA Prompt Loader

Load the saved FEA prompt from disk and inspect the exact text that will be used later.


In [ ]:
from pathlib import Path
import json
import logging
import os
import sys

import yaml
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

API_ENV_PATH = MODULE_ROOT / "src" / "api.env"

def _load_api_env(path: Path) -> dict[str, str]:
    env_values: dict[str, str] = {}
    if not path.exists():
        return env_values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("'").strip('"')
        if key and value:
            env_values[key] = value
            os.environ.setdefault(key, value)
    return env_values

api_env_values = _load_api_env(API_ENV_PATH)

from src import interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

FIXED_SAMPLE_PATH = MODULE_ROOT / "experiment_config" / "fixed_sample.yaml"
OUTPUT_ROOT = MODULE_ROOT / "outputs"
sample_config = yaml.safe_load(FIXED_SAMPLE_PATH.read_text(encoding="utf-8"))
sample_id = str(sample_config["sample_id"])
selection_source = str(sample_config.get("load_source", "dataset"))
state_b_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "02_fea_constrained_revision"

print("[STAGE] Setup")
print(f"  → sample id      : {sample_id}")
print(f"  → selection mode : {selection_source}")
print(f"  → state B dir : {state_b_dir}")
print(f"  → api.env     : {API_ENV_PATH}")
print(f"  → api.env keys: {sorted(api_env_values)}")


In [2]:
print("[STAGE] Load saved prompt")
prompt_path = state_b_dir / "fea_revision_prompt.txt"
load_case_path = state_b_dir / "load_case.json"
selector_hints_path = state_b_dir / "selector_hints.json"

assert prompt_path.exists(), f"missing prompt file: {prompt_path}"
assert load_case_path.exists(), f"missing load case: {load_case_path}"
assert selector_hints_path.exists(), f"missing selector hints: {selector_hints_path}"

fea_prompt = prompt_path.read_text(encoding="utf-8")
load_case = json.loads(load_case_path.read_text(encoding="utf-8"))
selector_hints = json.loads(selector_hints_path.read_text(encoding="utf-8"))

print(f"  → prompt file   : {prompt_path}")
print(f"  → prompt lines  : {len(fea_prompt.splitlines())}")
print(f"  → load case keys : {sorted(load_case.keys())}")
print(f"  → hint keys      : {sorted(selector_hints.keys())}")

display(Markdown("## Saved FEA prompt"))
display(Markdown(f"```text\n{fea_prompt}\n```"))


[STAGE] Load saved prompt
  → prompt file   : /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/02_fea_constrained_revision/fea_revision_prompt.txt
  → prompt lines  : 114
  → load case keys : ['boundary_conditions', 'loads', 'material', 'requirements', 'sample_id', 'units']
  → hint keys      : ['fixed_region_description', 'fixed_region_selector', 'load_region_description', 'load_region_selector', 'notes', 'sample_id']


## Saved FEA prompt

```text
Revise the original DB CadQuery design with FEA constraints.

State A original prompt:
Write Python code using CadQuery imported as cq.
declare diameter = 0.321427, cylinder_length = 0.75025, sides = 6, length = 0.61859, extrude = 0.214286
create cyl workplane "XZ" cylinder_length, diameter/2
cyl faces "-Y" polygon sides, length workplane().extrude
translate y axis cylinder_length/2

State A original DB code:
```python
import cadquery as cq

diameter:float = 0.321427
cylinder_length:float = 0.75025

sides:int = 6
length:float = 0.61859
extrude:float = 0.214286

cylinder = cq.Workplane("XZ").cylinder(cylinder_length, diameter/2)
part:cq.Workplane = cylinder.faces("-Y").polygon(sides, length).workplane().extrude(extrude)

part = part.translate((0,cylinder_length/2,0))

cq.exporters.export(part, 'Ground_Truth.stl')
```

Load case (JSON):
{
  "boundary_conditions": [
    {
      "description": "User-defined or model-inferred fixed support region",
      "id": "fixed_region",
      "selector": null,
      "type": "fixed_displacement"
    }
  ],
  "loads": [
    {
      "description": "User-defined or model-inferred load application region",
      "direction": [
        0,
        0,
        -1
      ],
      "id": "load_region",
      "magnitude_n": 200,
      "selector": null,
      "type": "force"
    }
  ],
  "material": {
    "name": "Aluminum 6061-T6",
    "poissons_ratio": 0.33,
    "yield_strength_pa": 276000000,
    "youngs_modulus_pa": 68900000000
  },
  "requirements": {
    "max_displacement_mm": 1.0,
    "max_von_mises_pa": 138000000,
    "required_safety_factor": 2.0
  },
  "sample_id": "00689964",
  "units": "mm"
}

Selector hints (JSON):
{
  "fixed_region_description": "wall-facing mounting plate face",
  "fixed_region_selector": {
    "axis": "x",
    "side": "minimum"
  },
  "load_region_description": "top face near free end",
  "load_region_selector": {
    "axis": "x",
    "side": "maximum"
  },
  "notes": [
    "Confirm the fixed region before running FreeCAD FEM.",
    "Confirm the load region before running FreeCAD FEM."
  ],
  "sample_id": "00689964"
}

Required content checklist:
- original prompt
- original DB code
- load case
- selector hints
- preserve identity
- permitted modifications
- machine-readable change log

State B revision instructions:
- Preserve the original design identity.
- Revise State A instead of designing an unrelated part.
- Keep the same functional intent, silhouette, and mounting logic where possible.
- Use only permitted modifications such as thickness changes, ribs, gussets, fillets, local strengthening, and support/load face cleanup.
- Keep the geometry as one connected solid when possible.
- Keep the result meshable and manufacturable.
- Do not introduce decorative or unrelated features.

Machine-readable change-log instructions:
- Return a machine-readable change_log object.
- Each change_log entry must describe the changed feature, change type, reason, and expected physical effect.
- Record whether the original design identity was preserved.
- Return only JSON containing code_lines and change_log.

Additional output requirements:
- Prompt output requirement: include original prompt, original DB code, load case, selector hints, identity-preservation rules, and machine-readable change-log instructions.
- Code output requirement: return runnable CadQuery Python code using import cadquery as cq and a result variable.
- Change-log requirement: describe modifications in a machine-readable structure.
```

In [3]:
print("[STAGE] Summary")
print(f"Sample ID: {sample_id}")
print(f"Prompt path: {prompt_path}")
print(f"Load case path: {load_case_path}")
print(f"Selector hints path: {selector_hints_path}")
print("✓ Saved prompt is readable and ready for downstream notebooks")


[STAGE] Summary
Sample ID: 00689964
Prompt path: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/02_fea_constrained_revision/fea_revision_prompt.txt
Load case path: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/02_fea_constrained_revision/load_case.json
Selector hints path: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/02_fea_constrained_revision/selector_hints.json
✓ Saved prompt is readable and ready for downstream notebooks


## What this notebook proves

- The saved FEA prompt can be loaded later.
- The prompt text matches the generated artifact.
- The prompt is ready for downstream notebooks.
